In [186]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [187]:
import sys
sys.path.append('/home/jovyan/work/base_demo')
import base_tool

In [188]:
import pandas as pd
pd.set_option('display.max_rows', 15)
pd.set_option('display.max_columns', 65)

数据介绍

bid_book_begin 集合竞价后的完整委托买入订单簿

ask_book_begin 集合竞价后的完整委托卖出订单簿

snap_list 连续竞价阶段的1s快照
    time_hms  时分秒字符串
    time_mark 毫秒级时间戳
    price_open 快照内首个成交价(无成交时为0.0)
    price_low  快照内最低成交价(无成交时为0.0)
    price_high 快照内最高成交价(无成交时为0.0)
    price_last 当日内最新成交价
     buy_trade 主动买入成交
    sell_trade 主动卖出成交
    bid_insert 委托买入挂单
    ask_insert 委托卖出挂单
    bid_cancel 委托买入撤单
    ask_cancel 委托卖出撤单

511520 511090 518880

In [ ]:
import numpy as np

class TrainValidTest():
    def __init__(self, snap_list, param_dict,feature_func,y_func):
        if param_dict is not None:
            self.__dict__.update(param_dict)
        # 确保必要属性存在
        if not hasattr(self, 'x_window'):
            self.x_window = 1
        if not hasattr(self, 'y_window'):
            self.y_window = 1

        self.snap_list = snap_list.copy()
        self.create_feature = feature_func
        self.create_y = y_func 
        self.trigger = self.trigger

        prices = pd.Series([snap['price_last'] for snap in snap_list])
        self.short_ma = prices.rolling(window=self.short_window, min_periods=1).mean()
        self.long_ma = prices.rolling(window=self.long_window, min_periods=1).mean()

    def samples(self):
        X_list, y_list = [], []
        n = len(self.snap_list)
        stride = getattr(self, 'stride', 1)
        category = 0
        for i in range(self.x_window, n - self.y_window, stride):
            flag , category = self.trigger(self.snap_list,i)
            if flag:
                x = self.create_feature(self.snap_list[i - self.x_window:i])
                volatility = x['volatility'].iloc[0] if 'volatility' in x.columns else 0.0
                y = self.create_y(self.snap_list[i:i + self.y_window], volatility, self.k_up, self.k_down,category)
                X_list.append(x)
                y_list.append(y)

        X_all = pd.concat(X_list, axis=0, ignore_index=True)
        y_all = pd.concat(y_list, axis=0, ignore_index=True)

        return X_all, y_all
    
    def trigger(self,time):
        short_curr = self.short_ma.iloc[time]
        long_curr = self.long_ma.iloc[time]
        short_prev = self.short_ma.iloc[time-1]
        long_prev = self.long_ma.iloc[time-1]


        if short_curr <= long_curr and short_prev > long_prev:
            return True,1
        elif short_curr >= long_curr and short_prev < long_prev:
            return True,-1
        else:      
            return False,0

def samples_from_dates(dates, instrument_id, param_dict, create_feature, create_y):
    X_all_list = []
    y_all_list = []
    
    for date in dates:
        try:
            snap_list = base_tool.snap_list_load(instrument_id, date)
            if len(snap_list) < param_dict['x_window'] + param_dict['y_window']:
                print(f"{date}: 数据不足，跳过")
                continue
            tv = TrainValidTest(snap_list, param_dict, create_feature, create_y)
            X_day, y_day = tv.samples()
            X_all_list.append(X_day)
            y_all_list.append(y_day)
            print(f"{date}: 产生 {len(X_day)} 个样本")
        except Exception as e:
            print(f"{date}: 加载失败 - {e}")
    
    if X_all_list:
        X_total = pd.concat(X_all_list, axis=0, ignore_index=True)
        y_total = pd.concat(y_all_list, axis=0, ignore_index=True)
    else:
        X_total = pd.DataFrame()
        y_total = pd.Series()
    
    return X_total, y_total

def create_y(snap_slice, volatility, k_up, k_down,category):
    # 初始化突破时间索引为 None
    t_up = None
    t_down = None

    start = snap_slice[0]['price_last']
    if start is None or start == 0 or pd.isna(start):
        return pd.Series([0])

    up = start * (1 + volatility * k_up)
    down = start * (1 - volatility * k_down)

    for i in range(1, len(snap_slice)):
        price = snap_slice[i]['price_last']
        if price is None or pd.isna(price):
            continue

        if t_up is None and price >= up:
            t_up = i
        if t_down is None and price <= down:
            t_down = i

        if t_up is not None and t_down is not None:
            break

    # 根据触发情况决定标签
    if t_up is not None and t_down is not None:
        label = 1 if t_up < t_down else -1
    elif t_up is not None:
        label = 1
    elif t_down is not None:
        label = -1
    else:
        label = 0

    if category == label:
        label = 1
    else:
        label = 0

    return pd.Series([label])

def create_feature(snap_slice):
    """
    从特征窗口快照切片中提取特征，并计算波动率。
    """
    last = snap_slice[-1]

    price = last['price_last']
    trades = last['num_trades']
    best_bid = last['bid_book'][0][0] if last['bid_book'] is not None and len(last['bid_book']) > 0 else np.nan
    best_ask = last['ask_book'][0][0] if last['ask_book'] is not None and len(last['ask_book']) > 0 else np.nan
    best_bid_depth = last['bid_book'][0][1] if last['bid_book'] is not None and len(last['bid_book']) > 0 else 0
    best_ask_depth = last['ask_book'][0][1] if last['ask_book'] is not None and len(last['ask_book']) > 0 else 0
    spread = best_ask - best_bid if not np.isnan(best_ask) and not np.isnan(best_bid) else np.nan

    bid_depth = sum([level[1] for level in last['bid_book']]) if last['bid_book'] is not None else 0
    ask_depth = sum([level[1] for level in last['ask_book']]) if last['ask_book'] is not None else 0

    OBI = (bid_depth - ask_depth) / (bid_depth + ask_depth) if (bid_depth + ask_depth) > 0 else 0

    WAMP = (best_bid_depth * best_bid + best_ask_depth * best_ask) / (best_bid_depth + best_ask_depth) if (bid_depth + ask_depth) > 0 else np.nan


    # 基于特征窗口内的价格序列计算波动率（相对波动率 = 标准差 / 均值）
    prices = [snap['price_last'] for snap in snap_slice if snap['price_last'] is not None]
    if len(prices) >= 2:
        mean_price = np.mean(prices)
        std_price = np.std(prices)
        volatility = std_price / mean_price if mean_price != 0 else 0.0
    else:
        volatility = 0.0

    features = {
        'price_last': price,
        'num_trades': trades,
        'best_bid': best_bid,
        'best_ask': best_ask,
        'spread': spread,
        'bid_depth1': best_bid_depth,
        'ask_depth1': best_ask_depth,
        'volatility': volatility,
        'OBI': OBI,
        'WAMP': WAMP,
        
    }
    return pd.DataFrame([features])



In [190]:
from collections import deque
import os
from typing import Dict, Any

class StrategyDemo():
    def __init__(self, model, param_dict={}) -> None:
        self.__dict__.update(param_dict)
        # 使用更具体的变量名
        data_file = f'/home/jovyan/work/backtest_result/{self.instrument_id}_{self.trade_ymd}_{self.name}.pkl'
        
        # 健壮的文件删除操作
        try:
            if os.path.exists(data_file):
                os.remove(data_file)
        except OSError as e:
            # 记录日志，而不是中断程序
            print(f"Warning: Could not delete file {data_file}: {e}")

        self.position_last = 0
        self.model = model
        
        # 优化1: 使用deque管理特征缓冲区
        self.feature_buffer = deque(maxlen=self.x_window)

        self.price_list = deque(maxlen=self.long_window)
        self.prev_signal = 0
        
        # 优化2: 引入增量计算所需的变量
        self.price_sum = 0.0

        return

    def on_snap(self, snap: Dict[str, Any]) -> None:
        price = snap.get('price_last') # 使用 .get() 方法更安全

        if not price: # 更Pythonic的空值检查
            return

        # 优化3: 增量更新价格总和
        if len(self.price_list) == self.x_window:
            # 如果缓冲区已满，减去即将被挤出的最旧价格
            self.price_sum -= self.price_list[0]
        
        self.price_list.append(price)
        self.price_sum += price

        if len(self.price_list) < self.x_window:
            self.position_last = 0
            self.prev_signal = 0
            return

        # 优化4: 利用增量总和计算均线，避免重复sum()
        long_ma = self.price_sum / self.long_window
        
        # 计算短期均线，由于short_window通常较小，这里影响不大
        # 但也可以用类似增量方式优化
        short_ma = sum(list(self.price_list)[-self.short_window:]) / self.short_window
        diff = short_ma - long_ma

        # 优化5: deque自动处理append和pop
        self.feature_buffer.append(snap)

        # 只有当特征缓冲区也准备好时才进行预测
        if len(self.feature_buffer) == self.x_window:
            features = create_feature(self.feature_buffer)
            prob = self.model.predict_proba(features)[:, 1][0]
        else:
            # 如果特征不足，可以设置一个默认prob或跳过
            return

        # 信号生成逻辑保持不变
        if diff > 0: # 金叉
            current_signal = 1
        elif diff < 0: # 死叉
            current_signal = -1
        else :
            current_signal = 0

        if current_signal != self.prev_signal and prob > self.threshold: 
            self.position_last = current_signal
            self.prev_signal = current_signal

        return

In [191]:
instrument_id = '518880'
trade_ymd = '20260319'

In [192]:
param_dict = {

    'name' : 'bollingger_bands',
    'instrument_id' : instrument_id,
    'trade_ymd' : trade_ymd,
    
    'short_window' : 180 ,
    'long_window' : 360 , 
    'threshold' : 0.00 ,
    'name' : 'MA_v1',

    'y_window' : 300 ,
    'stride': 1,

    'k_up': 3,
    'k_down': 3
}
param_dict['x_window'] = max(param_dict['short_window'],param_dict['long_window'])


In [193]:
import xgboost as xgb
import joblib
from sklearn.metrics import accuracy_score, classification_report

instrument_id = '511520'
train_days = 20
valid_days = 4
test_days = 5

trade_dates = ['20260202', '20260203', '20260204', '20260205', '20260206',
                '20260209', '20260210', '20260211', '20260212', '20260213',
                '20260224', '20260225', '20260226', '20260227', '20260302',
                '20260303', '20260304', '20260305', '20260306', '20260309',
                '20260310', '20260311', '20260312', '20260313', '20260316',
                '20260317', '20260318', '20260319', '20260320']
print(f"总交易日数量: {len(trade_dates)}")
print(f"交易日范围: {trade_dates[0]} ~ {trade_dates[-1]}")

总交易日数量: 29
交易日范围: 20260202 ~ 20260320


In [194]:
train_dates = trade_dates[:train_days]
valid_dates = trade_dates[train_days:train_days + valid_days]
test_dates = trade_dates[train_days + valid_days:train_days + valid_days + test_days]

print(f"训练集: {train_dates[0]} ~ {train_dates[-1]} ({len(train_dates)}天)")
print(f"验证集: {valid_dates[0]} ~ {valid_dates[-1]} ({len(valid_dates)}天)")
print(f"测试集: {test_dates[0]} ~ {test_dates[-1]} ({len(test_dates)}天)")

训练集: 20260202 ~ 20260309 (20天)
验证集: 20260310 ~ 20260313 (4天)
测试集: 20260316 ~ 20260320 (5天)


In [195]:
print("生成训练集样本...")
X_train, y_train = samples_from_dates(train_dates, instrument_id, param_dict, create_feature, create_y)
print(f"训练集样本: X={X_train.shape}, y={y_train.shape}")
if len(y_train) > 0:
    print(f"标签分布:\n{y_train.value_counts()}")

生成训练集样本...


20260202: 产生 39 个样本
20260203: 产生 44 个样本
20260204: 产生 37 个样本
20260205: 产生 44 个样本
20260206: 产生 46 个样本
20260209: 产生 48 个样本
20260210: 产生 38 个样本
20260211: 产生 41 个样本
20260212: 产生 30 个样本


KeyboardInterrupt: 

In [ ]:
%%time
print("生成验证集样本...")
X_valid, y_valid = samples_from_dates(valid_dates, instrument_id, param_dict, create_feature, create_y)
print(f"验证集样本: X={X_valid.shape}, y={y_valid.shape}")
if len(y_valid) > 0:
    print(f"标签分布:\n{y_valid.value_counts()}")

生成验证集样本...
20260310: 产生 479 个样本
20260311: 产生 505 个样本
20260312: 产生 467 个样本
20260313: 产生 502 个样本
验证集样本: X=(1953, 10), y=(1953,)
标签分布:
0    1152
1     801
Name: count, dtype: int64
CPU times: user 20.2 s, sys: 3.01 ms, total: 20.2 s
Wall time: 20.2 s


In [ ]:
%%time
model = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=3,
    learning_rate=0.01,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='binary:logistic',
    random_state=42,
    n_jobs=-1,
    verbosity=1
)

model.fit(
    X_train, y_train,
    eval_set=[(X_valid, y_valid)],
    verbose=10
)

[0]	validation_0-logloss:0.67973
[10]	validation_0-logloss:0.67900
[20]	validation_0-logloss:0.67843


[30]	validation_0-logloss:0.67791
[40]	validation_0-logloss:0.67752
[50]	validation_0-logloss:0.67723
[60]	validation_0-logloss:0.67697
[70]	validation_0-logloss:0.67678
[80]	validation_0-logloss:0.67656
[90]	validation_0-logloss:0.67647
[100]	validation_0-logloss:0.67629
[110]	validation_0-logloss:0.67615
[120]	validation_0-logloss:0.67596
[130]	validation_0-logloss:0.67580
[140]	validation_0-logloss:0.67567
[150]	validation_0-logloss:0.67557
[160]	validation_0-logloss:0.67553
[170]	validation_0-logloss:0.67547
[180]	validation_0-logloss:0.67540
[190]	validation_0-logloss:0.67534
[199]	validation_0-logloss:0.67534
CPU times: user 29.8 s, sys: 32 ms, total: 29.8 s
Wall time: 1.03 s


,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes f

# 0.55
# 0.58

In [ ]:
y_pred_encoded = model.predict(X_valid)
y_pred = le.inverse_transform(y_pred_encoded)
print("验证集准确率:", accuracy_score(y_valid, y_pred))

unique_labels = sorted(set(y_valid.unique()) | set(np.unique(y_pred)))
label_names = { 0: '不可信(0)', 1: '可信(1)' }
target_names = [label_names.get(l, str(l)) for l in unique_labels]
print("分类报告:")
print(classification_report(y_valid, y_pred, labels=unique_labels, target_names=target_names))

验证集准确率: 0.5898617511520737
分类报告:
              precision    recall  f1-score   support

      不可信(0)       0.59      1.00      0.74      1152
       可信(1)       0.00      0.00      0.00       801

    accuracy                           0.59      1953
   macro avg       0.29      0.50      0.37      1953
weighted avg       0.35      0.59      0.44      1953



/home/jovyan/miniconda3/envs/Quant/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/jovyan/miniconda3/envs/Quant/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/jovyan/miniconda3/envs/Quant/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.c

In [ ]:
model_path = f'/home/jovyan/work/backtest_result/{instrument_id}_xgb_model.pkl'
joblib.dump(model, model_path)
print(f"模型已保存至: {model_path}")


模型已保存至: /home/jovyan/work/backtest_result/511520_xgb_model.pkl


# 测试集表现

In [ ]:
%%time
print("生成测试集样本...")
X_test, y_test = samples_from_dates(test_dates, instrument_id, param_dict, create_feature, create_y)
print(f"测试集样本: X={X_test.shape}, y={y_test.shape}")

生成测试集样本...
20260316: 产生 521 个样本
20260317: 产生 505 个样本
20260318: 产生 551 个样本
20260319: 产生 522 个样本
20260320: 产生 495 个样本
测试集样本: X=(2594, 10), y=(2594,)
CPU times: user 26.4 s, sys: 5.01 ms, total: 26.4 s
Wall time: 26.4 s


In [ ]:
import os
import sys
current_notebook_path = os.path.abspath('%pwd') 
current_dir = os.path.dirname(current_notebook_path)
parent_dir = os.path.dirname(current_dir)
utils_path = os.path.join(parent_dir, 'tools')
sys.path.append(utils_path)

In [ ]:
from backtesting import backtest_multi_days
param_dict['trade_ymd'] = ''
strategy = StrategyDemo(model,param_dict)
backtest_df = backtest_multi_days(instrument_id,'20260316','20260320',strategy,param_dict)

KeyboardInterrupt: 

In [ ]:
backtest_df

,trade_ymd,order_time,order_price,total,trade,cancel,hold,profit_last,profits,maxdd,MAR,pper
0,20260317,2026-03-17 14:55:00,115.895,698,692,6,0,0.0,-64.8,64.8,-1.00,-0.09
1,20260318,2026-03-18 14:55:00,116.031,764,749,15,0,-0.4,-74.8,74.8,-1.00,-0.10
2,20260319,2026-03-19 14:55:00,116.169,666,652,14,0,-0.3,-70.4,70.7,-1.00,-0.11
3,20260320,2026-03-20 14:55:00,116.062,673,656,17,0,-0.2,-63.6,67.1,-0.95,-0.10


In [ ]:
from backtesting import backtest_summary
summary = backtest_summary(backtest_df)
print(summary)

{'交易天数': 4, '累计盈亏': np.float64(-273.6), '最大单日盈利': -63.6, '最大单日亏损': -74.8, '盈利天数': 0, '亏损天数': 4, '平盘天数': 0, '胜率(%)': np.float64(0.0), '日均盈亏': np.float64(-68.4), '盈亏比': np.float64(0.0)}
